In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# MileStone-1

In [ ]:
####################################### Importing
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
##############################################
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"# Enter dataset path
GENRES = [
 'blues', 'classical', 'country', 'disco', 'hiphop',
 'jazz', 'metal', 'pop', 'reggae', 'rock'
] # Make the list of all genres available (alphabetical order)
STEMS = {
    "drums.wav":  "drums",
    "vocals.wav": "vocals",
    "bass.wav":   "bass",
    "other.wav": "other"
}

 # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [ ]:
##############################################################
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)

        # Check: if genre folder exists
        if not os.path.isdir(genre_path):
            continue

        song_dirs = sorted(os.listdir(genre_path))
        valid_songs = []

        # Iterate through songs
        for song in song_dirs:
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            stem_paths = {}
            valid = True

            # Completeness + corruption check
            for stem_file, stem_key in STEMS.items():
                fp = os.path.join(song_path, stem_file)

                if not os.path.isfile(fp) or os.path.getsize(fp) < 4096:
                    valid = False
                    break

                stem_paths[stem_key] = fp

            if valid:
                valid_songs.append(stem_paths)

        # Stratified shuffle split (within genre)
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_key, path in song.items():
                    target_dict[genre][stem_key].append(path)

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset
tr, val = build_dataset(DATA_ROOT)

In [ ]:
#####################################################
#Question-1
import os

# Size thresholds
KB = 1024
MB = 1024 * 1024

CORRUPTED_LIMIT = 4 * KB                  # 4 KB
SMALL_SOUND_LIMIT = 5.0491 * MB     # 5.0491 MB

def count_small_and_corrupted_sounds(root_dir):
    corrupted_count = 0
    small_sound_count = 0

    for genre in os.listdir(root_dir):
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            for file in os.listdir(song_path):
                if not file.endswith(".wav"):
                    continue

                file_path = os.path.join(song_path, file)

                try:
                    size = os.path.getsize(file_path)
                except OSError:
                    continue

                if size < CORRUPTED_LIMIT:
                    corrupted_count += 1

                if size < SMALL_SOUND_LIMIT:
                    small_sound_count += 1

    total = corrupted_count + small_sound_count
    return corrupted_count, small_sound_count, total


# Run
corrupted, small, total = count_small_and_corrupted_sounds(DATA_ROOT)

print("Corrupted sounds (< 4 KB):", corrupted)
print("Small sounds (< 5.0491 MB):", small)
print("Total (corrupted + small):", total)


In [ ]:
########################################################
#Question-2
import os

# Size units
KB = 1024
MB = 1024 * 1024

# Thresholds
UPPER_LIMIT = 5.0493 * MB   # > 5.0493 MB
LOWER_LIMIT = 5.0491 * MB   # < 5.0491 MB

def absolute_difference_large_vs_small(root_dir):
    greater_count = 0
    smaller_count = 0

    for genre in os.listdir(root_dir):
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            for file in os.listdir(song_path):
                if not file.endswith(".wav"):
                    continue

                file_path = os.path.join(song_path, file)

                try:
                    size = os.path.getsize(file_path)
                except OSError:
                    continue

                if size > UPPER_LIMIT:
                    greater_count += 1
                elif size < LOWER_LIMIT:
                    smaller_count += 1

    return abs(greater_count - smaller_count), greater_count, smaller_count


# Run
abs_diff, greater, smaller = absolute_difference_large_vs_small(DATA_ROOT)

print("Sounds > 5.0493 MB:", greater)
print("Sounds < 5.0491 MB:", smaller)
print("Absolute difference:", abs_diff)

In [ ]:
####################################################
#Question-3
# Assuming these are already created
# tr, val = build_dataset(DATA_ROOT)

# Count samples
train_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])

# Absolute difference
abs_diff = abs(train_reggae_drums - val_country_vocals)

print("Training reggae drum samples:", train_reggae_drums)
print("Validation country vocal samples:", val_country_vocals)
print("Absolute difference:", abs_diff)


In [ ]:
#################################################################
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    # ---- COUNT TOTAL FILES ----
    total_files = sum(len(paths) for genre_data in dataset_dict.values() 
                     for paths in genre_data.values())
    
    print(f"Analyzing {total_files} files for long silences...")
    
    # Create progress bar
    pbar = tqdm(total=total_files, desc="Processing files")
    
    for genre, stems_dict in dataset_dict.items():
        for stem_name, file_paths in stems_dict.items():
            for file_path in file_paths:
                try:
                    # Load Audio
                    y, _ = librosa.load(file_path, sr=sr, duration=None)
                    total_duration = len(y) / sr
                    
                    # Find Non-Silent Intervals
                    non_silent_intervals = librosa.effects.split(y, top_db=top_db)
                    
                    max_silence = 0.0
                    silence_type = []
                    
                    if len(non_silent_intervals) == 0:
                        # CASE A: Fully silent
                        max_silence = total_duration
                        silence_type = ["start", "middle", "end"]
                    else:
                        # CASE B: START silence
                        start_silence = non_silent_intervals[0][0] / sr
                        if start_silence > max_silence:
                            max_silence = start_silence
                            silence_type = ["start"]
                        
                        # CASE C: END silence
                        end_silence = (len(y) - non_silent_intervals[-1][1]) / sr
                        if end_silence > max_silence:
                            max_silence = end_silence
                            silence_type = ["end"]
                        elif abs(end_silence - max_silence) < 0.01:  # Same as current max
                            if "end" not in silence_type:
                                silence_type.append("end")
                        
                        # CASE D: MIDDLE silence
                        for i in range(len(non_silent_intervals) - 1):
                            gap_start = non_silent_intervals[i][1]
                            gap_end = non_silent_intervals[i + 1][0]
                            gap_duration = (gap_end - gap_start) / sr
                            
                            if gap_duration > max_silence:
                                max_silence = gap_duration
                                silence_type = ["middle"]
                            elif abs(gap_duration - max_silence) < 0.01:
                                if "middle" not in silence_type:
                                    silence_type.append("middle")
                    
                    # Store result
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })
                
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                
                pbar.update(1)
    
    pbar.close()
    #-------------------------------------------------------------------------
    df = pd.DataFrame(records)
    return df 

In [ ]:
######################################################
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
pivot = pd.pivot_table(
    df_silence,
    index="Genre",
    columns="Stem",
    values="File_Path",
    aggfunc="count",
    fill_value=0
)

print(pivot)


In [ ]:
###########################################
#Question-4


print("Total training files with silence ≥ 5 sec:", len(df_silence))

In [ ]:
##############################################################
#Question-5
total_vocals_silence = (df_silence["Stem"] == "vocals").sum()

print("Total number of vocals sound tracks with silence >= 5 secs:",
      total_vocals_silence)

In [ ]:
###########################################
#question-6
avg_vocals_silence = (
    df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"]
    .mean()
)

print("Average Silence Length in Vocals (secs):", round(avg_vocals_silence, 2))


In [ ]:
#########################################
#question-7
count_jazz_drums = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
].shape[0]

print("Total jazz drum tracks with silence ≥ 5 sec:", count_jazz_drums)


In [ ]:
#################################################
#Question-8
count_jazz_drums_middle = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle" )
].shape[0]

print("Total jazz drum tracks with silence ≥ 5 sec ONLY in middle:",
      count_jazz_drums_middle)


In [ ]:
###################################################
#Question-9
count = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]

print("Total number of jazz drum tracks with silence ≥ 5s and max silence ≥ 10s:", count)

In [ ]:
###########################################
#
stems_audio = []

try:
    for key in STEM_KEYS:
        # Get list of files for this genre & stem
        file_list = tr[GENRE_TO_TEST][key]

        # Pick SONG_INDEX-th song
        file_path = file_list[SONG_INDEX]

        # ------------------- LOAD AUDIO -------------------------------
        # Load only first 5 seconds for speed/consistency
        y, _ = librosa.load(file_path, sr=SR, duration=5.0)
        stems_audio.append(y)
        # ---------------------------------------------------------------

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print("Unexpected error:", e)

In [ ]:
##########################################################################
#
import numpy as np

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.stack(stems_audio, axis=0)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

print("RMS amplitude:", rms_val) #Question-11
print("Peak value after normalization:", np.max(np.abs(mix_norm)))#Question-12

In [ ]:
#####################################################
#Question-10
mix_length_sec = len(mix_raw)

print("Length of mix sample (seconds):", round(mix_length_sec, 2))

In [ ]:
########################## dummy ##############
data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")
data.to_csv("submission.csv",index=False)

# Milestone-2

In [ ]:
#######################################################################
#Question-1
durations = []

for stem, file_paths in tr["jazz"].items():
    for file_path in file_paths:
        try:
            y, sr = librosa.load(file_path, sr=SR)
            duration = len(y) / sr
            durations.append(duration)
        except Exception:
            continue

mean_duration = np.mean(durations)

print("Mean duration of Jazz stems (seconds):", round(mean_duration, 2))

In [ ]:
########################################################
#Question-2

sample_rates = set()

for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, genre)
    
    if not os.path.isdir(genre_path):
        continue
    
    for song in sorted(os.listdir(genre_path)):
        song_path = os.path.join(genre_path, song)
        
        if not os.path.isdir(song_path):
            continue
        
        for stem_file in STEMS.keys():
            file_path = os.path.join(song_path, stem_file)
            
            if os.path.exists(file_path):
                try:
                    sr = librosa.get_samplerate(file_path)
                    sample_rates.add(sr)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

sample_rates = sorted(sample_rates)
print(sample_rates)

In [ ]:
############################################################
#Question-3
#as Bulid_data fuction gives valid song files
print(0)

In [ ]:
#######################################################################
#Question-4
from tqdm import tqdm

peak_db_values = []

for genre, stems_dict in tr.items():
    vocal_files = stems_dict.get("vocals", [])
    
    for file_path in tqdm(vocal_files, desc=f"Processing {genre} vocals"):
        try:
            y, _ = librosa.load(file_path, sr=None)
            
            peak = np.max(np.abs(y))
            
            if peak > 0:
                peak_db = 20 * np.log10(peak)
                peak_db_values.append(peak_db)
        
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

# Compute average
avg_peak_db = np.mean(peak_db_values)

print(f"Average peak amplitude (dB) for vocal stems in train dataset: {avg_peak_db:.2f} dB")

In [ ]:
##############################################################################
#Question-5
blues_path = os.path.join(DATA_ROOT, "blues")

centroids = []

for song in os.listdir(blues_path):
    song_path = os.path.join(blues_path, song)
    
    for fname in ['other.wav', 'others.wav']:
        file_path = os.path.join(song_path, fname)
        
        if os.path.exists(file_path):
            try:
                y, sr = librosa.load(file_path, sr=22050)
                
                if len(y) > 0:
                    spec_cent = np.mean(
                        librosa.feature.spectral_centroid(y=y, sr=sr)
                    )
                    centroids.append(spec_cent)
            except:
                continue

print("Answer(Mean Spectral Centroid Blues):", np.mean(centroids))

In [ ]:
###################################################################
#Question-6
genre_centroids = {}

for g in GENRES:
    centroids = []
    genre_path = os.path.join(DATA_ROOT, g)
    
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        
        for fname in ['other.wav', 'others.wav']:
            file_path = os.path.join(song_path, fname)
            
            if os.path.exists(file_path):
                try:
                    y, sr = librosa.load(file_path, sr=22050)
                    
                    if len(y) > 0:
                        spec_cent = np.mean(
                            librosa.feature.spectral_centroid(y=y, sr=sr)
                        )
                        centroids.append(spec_cent)
                except:
                    continue
    
    if len(centroids) > 0:
        genre_centroids[g] = np.mean(centroids)

print("All Genre Means:", genre_centroids)
print("Answer Q6 (Highest Centroid Genre):",
      max(genre_centroids, key=genre_centroids.get))

In [ ]:
#########################################################################
#Question-7
silence_count = 0

for g in GENRES:
    genre_path = os.path.join(DATA_ROOT, g)
    
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        
        for stem in os.listdir(song_path):
            file_path = os.path.join(song_path, stem)
            
            if os.path.exists(file_path):
                try:
                    y, sr = librosa.load(file_path, sr=None)
                    
                    first_half_sec = y[:int(0.5 * sr)]
                    
                    if np.max(np.abs(first_half_sec)) < 1e-4:
                        silence_count += 1
                        
                except:
                    continue

print("(Silence Count):", silence_count)

In [ ]:
###################################################################
def extract_features_safe(song_path):
    
    # auto detect stem name
    possible_files_1 = ['other.wav', 'others.wav']
    file_path = None
    
    for fname in possible_files_1:
        temp_path = os.path.join(song_path, fname)
        if os.path.exists(temp_path):
            file_path = temp_path
            break
    
    if file_path is None:
        return None
    
    try:
        y, sr = librosa.load(file_path, sr=22050, duration=10)
        
        if len(y) == 0:
            return None
        
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        
        return [float(tempo), spec_cent, zcr, rolloff]
    
    except:
        return None

In [ ]:
#######################################################################
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
data = []

for g in GENRES:
    gp = os.path.join(DATA_ROOT, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    
    for s in songs[:50]:   # speed ke liye
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['genre'],
    random_state=42
)

In [ ]:
###############################################################
X_train = []
y_train = []

for path, genre in zip(train_df['path'], train_df['genre']):
    features = extract_features_safe(path)
    if features is not None:
        X_train.append(features)
        y_train.append(genre)

X_val = []
y_val = []

for path, genre in zip(val_df['path'], val_df['genre']):
    features = extract_features_safe(path)
    if features is not None:
        X_val.append(features)
        y_val.append(genre)

X_train = np.array(X_train)
X_val = np.array(X_val)

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
#####################################################
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
##########################################################
#Question-8
y_pred = clf.predict(X_val)

macro_f1 = f1_score(y_val, y_pred, average='macro')
print("(Validation Macro F1):", macro_f1)


In [ ]:
####################################################
#Question-9
cr = classification_report(y_val, y_pred, output_dict=True)

print("(Precision hiphop):", cr['hiphop']['precision'])

In [ ]:
#########################################################
#Question-10
print("(Recall pop):", cr['pop']['recall'])

In [ ]:
##########################################################
#Question-11
accuracy = np.mean(y_pred == y_val)
print("(Accuracy):", accuracy)

In [ ]:
##############################################################
#Question-12
cm = confusion_matrix(y_val, y_pred, labels=GENRES)

tp_dict = {}

for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    tp_dict[genre] = TP

print("(Highest TP Genre):", max(tp_dict, key=tp_dict.get))

In [ ]:
############################################################
#Question-13
fn_dict = {}

for i, genre in enumerate(GENRES):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    fn_dict[genre] = FN

print("(Lowest FN Genre):", min(fn_dict, key=fn_dict.get))